In [1]:
import numpy as np
import keras
from keras import layers

In [2]:
max_features = 20000 # means use most 20000 common words from dataset
maxlen = 200 # only take first 200 words of each review

In [14]:
## Building the Model
inputs = keras.Input(shape = (maxlen,),dtype = "int32") # <KerasTensor shape=(None, 200), dtype=int32, sparse=False, ragged=False, name=keras_tensor>
x = layers.Embedding(max_features,128)(inputs)
x = layers.Bidirectional(layers.LSTM(64,return_sequences=True))(x)
x = layers.Bidirectional(layers.LSTM(64))(x)
outputs = layers.Dense(1,activation="sigmoid")(x)

In [16]:
model = keras.Model(inputs,outputs)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_5 (InputLayer)      │ (None, 200)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_2 (Embedding)         │ (None, 200, 128)       │     2,560,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 200, 128)       │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 128)            │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,757,761 (10.52 MB)

 Trainable params: 2,757,761 (10.52 MB)

 Non-trainable params: 0 (0.00 B)

In [17]:
(x_train,y_train),(x_val,y_val) = keras.datasets.imdb.load_data(num_words = max_features)

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [19]:
print(len(x_train),"Training Length")
print(len(x_val), "Validation Length")

25000 Training Length
25000 Validation Length


In [20]:
# make every review to 200 words
x_train = keras.utils.pad_sequences(x_train, maxlen=maxlen)
x_val   = keras.utils.pad_sequences(x_val, maxlen=maxlen)

In [21]:
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.fit(x_train, y_train, batch_size=32, epochs=2, validation_data=(x_val, y_val))

Epoch 1/2
782/782 ━━━━━━━━━━━━━━━━━━━━ 373s 470ms/step - accuracy: 0.7917 - loss: 0.4480 - val_accuracy: 0.8512 - val_loss: 0.3626
Epoch 2/2
782/782 ━━━━━━━━━━━━━━━━━━━━ 367s 451ms/step - accuracy: 0.8907 - loss: 0.2725 - val_accuracy: 0.8529 - val_loss: 0.3469


In [22]:
## Predict my reviews
# Get the word-to-number dictionary
word_index = keras.datasets.imdb.get_word_index()

1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [23]:
# write a function to convert reviews to numbers (embeddings)
def encode_review(text):
  words = text.lower().split()
  encoded = [word_index.get(word,2)+3 for word in words]
  padded = keras.utils.pad_sequences([encoded], maxlen=maxlen)
  return padded


In [29]:
# Write your custom reviews
my_reviews = [
    "This movie was absolutely fantastic! I loved every moment of it.",
    "Terrible film. Waste of time. Boring and predictable.",
    "Not bad at all, I quite enjoyed it.",
    "Not sure."
]

In [31]:
# Predict
for review in my_reviews:
  encoded = encode_review(review)
  score = model.predict(encoded,verbose=0)[0][0]
  if score<=0.40:
    sentiment = "NEGATIVE"
  elif score>0.40 and score <=0.6:
      sentiment = "NEUTRAL"
  else: sentiment = "POSITIVE"
  print(f"Review: {review}")
  print(f"Score: {score:.2f}  →  {sentiment}")
  print()

Review: This movie was absolutely fantastic! I loved every moment of it.
Score: 0.87  →  POSITIVE

Review: Terrible film. Waste of time. Boring and predictable.
Score: 0.05  →  NEGATIVE

Review: Not bad at all, I quite enjoyed it.
Score: 0.91  →  POSITIVE

Review: Not sure.
Score: 0.42  →  NEUTRAL

